In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Cytotoxic dataset integration and label-consistency analysis
- This notebook performs large-scale integration, harmonization, and curation of cytotoxic peptide annotations by aggregating multiple heterogeneous data sources into a single, sequence-centric representation.

- The inputs consist of independently curated cytotoxic peptide datasets derived from multiple public resources, including AMPDB v1, BIOPEP-UWM, CICERON, DRAMP, iAMPCN, MultiTox, and Peptipedia2.0. Each source provides sequence-level cytotoxic labels with varying coverage, redundancy, and annotation completeness.

- All peptide sequences associated with cytotoxic activity are first pooled across sources, and a global set of unique sequences is constructed. A pivot table is then created in which each row corresponds to a unique peptide sequence and each column represents a data source, enabling direct comparison of annotations across databases.

- Sequence-level quality control is applied prior to label integration. Sequences containing non-canonical amino acids are removed, and length-based filtering is enforced using globally defined minimum and maximum thresholds. These steps ensure biochemical consistency and compatibility with downstream machine learning pipelines.

- After filtering, cytotoxic labels from each source are mapped onto the pivot table using a unified encoding scheme (positive, negative, unlabeled, unknown). Label agreement is evaluated by computing per-sequence label counts, positive and negative vote percentages, and a set of high-level classification flags that identify sequences with consistent evidence (exclusive positive or exclusive negative), sequences lacking definitive labels, and sequences with conflicting annotations across sources.

- Ambiguous sequences are explicitly retained and further stratified according to the proportion of positive annotations, allowing downstream analyses to apply adjustable confidence thresholds rather than enforcing hard exclusion.

- The notebook produces curated, non-overlapping subsets of peptide sequences, including strictly cytotoxic sequences, strictly non-cytotoxic sequences, and ambiguous sequences with mixed evidence. In addition, a comprehensive metadata file is generated, summarizing source contributions, filtering statistics, sequence length distributions, and label agreement patterns across databases.

In [2]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/cytotoxic"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources

In [3]:
df_AMPDB_cytotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/AMPDB v1/processed_cytotoxic_dataset.csv")
df_AMPDB_cytotoxic = df_AMPDB_cytotoxic.rename(columns={"label": "cytotoxic"})

In [4]:
df_BIOPEP_UWM_cytotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/BIOPEP-UWM/processed_cytotoxic_dataset.csv")
df_BIOPEP_UWM_cytotoxic = df_BIOPEP_UWM_cytotoxic.rename(columns={"label": "cytotoxic"})

In [5]:
df_CICERON_cytotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/CICERON/processed_cytotoxic_dataset.csv")
df_CICERON_cytotoxic = df_CICERON_cytotoxic.rename(columns={"label": "cytotoxic"})

In [6]:
df_DRAMP_cytotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/DRAMP/processed_cytotoxic_dataset.csv")
df_DRAMP_cytotoxic = df_DRAMP_cytotoxic.rename(columns={"label": "cytotoxic"})

In [7]:
df_iAMPCN_cytotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/iAMPCN/processed_cytotoxic_dataset.csv")
df_iAMPCN_cytotoxic = df_iAMPCN_cytotoxic.rename(columns={"label": "cytotoxic"})

In [8]:
df_Multitox_cytotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Multitox/processed_cytotoxic_dataset.csv")
df_Multitox_cytotoxic = df_Multitox_cytotoxic.rename(columns={"label": "cytotoxic"})

In [9]:
df_peptipedia_cytotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/Peptipedia2.0/processed_cytotoxic_dataset.csv")
df_peptipedia_cytotoxic = df_peptipedia_cytotoxic.rename(columns={"label": "cytotoxic"})

- Collecting all sequences for activity

In [10]:
df_list_cytotoxic = [
    df_AMPDB_cytotoxic, df_BIOPEP_UWM_cytotoxic, df_CICERON_cytotoxic,
    df_DRAMP_cytotoxic, df_iAMPCN_cytotoxic, df_Multitox_cytotoxic, 
    df_peptipedia_cytotoxic
]
unique_sequence_cytotoxic = count_unique_sequence(df_list_cytotoxic)

27136


- Create pivote dataset

In [11]:
df_pivote = create_pivote(unique_sequence_cytotoxic)

- Removing sequences with non canonical residues 

In [12]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [13]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True     26856
False      280
Name: count, dtype: int64


- Filter sequences by length

In [14]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count    26856.000000
mean        76.541518
std        174.357072
min          2.000000
25%         15.000000
50%         23.000000
75%         44.000000
max       4545.000000
Name: length, dtype: float64

In [15]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [16]:
df_pivote["filter_length"].value_counts()

filter_length
True     22102
False     4754
Name: count, dtype: int64

In [17]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [18]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(22102, 4)

In [19]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [20]:
df_list_cytotoxic = [("AMPDB", df_AMPDB_cytotoxic),
                    ("BIOPEP-UWM", df_BIOPEP_UWM_cytotoxic),
                    ("CICERON", df_CICERON_cytotoxic),
                    ("DRAMP", df_DRAMP_cytotoxic),
                    ("iAMPCN", df_iAMPCN_cytotoxic),
                    ("MultiTox", df_Multitox_cytotoxic),
                    ("Peptipedia2.0", df_peptipedia_cytotoxic)
                ]

In [21]:
for source, dataset in df_list_cytotoxic:
    dataset = dataset[["sequence", "cytotoxic"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["cytotoxic"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

In [22]:
df_pivote.head(5)

,sequence,AMPDB,BIOPEP-UWM,CICERON,DRAMP,iAMPCN,MultiTox,Peptipedia2.0
0,LKKVYKRVARLIKRLFRYLKRPVR,999,999,999,999,0,999,999
2,RLGTRCSVCMLHAWQGGKQVDE,999,999,999,999,0,999,999
4,VALGPCYLQGTDPGASADAEGPQCPVTCTCSY,999,999,999,999,0,999,999
5,GLRKRLRKFRNKIKEKLKKEGQKIQGLLPKLAPRTDY,999,999,999,999,0,999,999
6,PKKINNTILKLLDRVASKI,999,999,999,999,0,999,999


- Working with pivote for detecting ambiguous sequences 

In [23]:
df_pivote = process_count_labels(df_pivote)

In [24]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
True     20434
False     1668
Name: count, dtype: int64

In [25]:
df_pivote["exclusive_0"].value_counts()

exclusive_0
True     20434
False     1668
Name: count, dtype: int64

In [26]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
False    21528
True       574
Name: count, dtype: int64

In [27]:
df_pivote["exclusive_1"].value_counts()

exclusive_1
False    21528
True       574
Name: count, dtype: int64

In [28]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    22102
Name: count, dtype: int64

In [29]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,AMPDB,BIOPEP-UWM,CICERON,DRAMP,iAMPCN,MultiTox,Peptipedia2.0,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
5669,WFKKIPKFLHLLKKF,999,999,999,999,999,999,1,1,0,0,6,True,False,True,False,False,0.0,100.0
12094,FLPIVAGLAANFLPKIVCKITKKC,999,999,999,999,1,999,1,2,0,0,5,True,False,True,False,False,0.0,100.0
22629,QLPICGETCVLGTCYTPGCSCAYPICVR,999,999,999,999,999,999,1,1,0,0,6,True,False,True,False,False,0.0,100.0
1260,AKVTMTCSAS,999,999,999,1,999,999,999,1,0,0,6,True,False,True,False,False,0.0,100.0
6727,ALWMTLLKKVLKAAAKAALNAVLVGANA,999,999,999,999,999,999,1,1,0,0,6,True,False,True,False,False,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9435,MKSAIAFVMVLAGLAVATESAPNNTPDLEARFCPVGKTCATDRECG...,999,999,999,999,0,999,999,0,1,0,6,False,True,False,True,False,100.0,0.0
9434,LVVAVTDGEADAAVEGLHDNTDFIHYGSHGKYPDNRPHGYPLD,999,999,999,999,0,999,999,0,1,0,6,False,True,False,True,False,100.0,0.0
9433,EPCTVGHRRYFTFGG,999,999,999,999,0,999,999,0,1,0,6,False,True,False,True,False,100.0,0.0
9431,AQRCGDQARGAKCPNCLCCGKYGFCGSGDAYCGEGSCQSQCRGCR,999,999,999,999,0,999,999,0,1,0,6,False,True,False,True,False,100.0,0.0


- Splitting data into only negative, only positive, and with amiguous data

In [30]:
negative = df_pivote[df_pivote["negative"]]

In [31]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [32]:
positive = df_pivote[df_pivote["positive"]]

In [33]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [34]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [35]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [36]:
df_ambiguous = categorize_percentage(df_ambiguous)

In [37]:
df_ambiguous["Category_pbb"].value_counts()

Category_pbb
40-50    931
30-40    133
60-70     30
Name: count, dtype: int64

- Working with metada

In [38]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="cytotoxic",
    source_list=df_list_cytotoxic,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)
metadata

{'task': 'cytotoxic',
 'generated_at': '2026-07-24T16:23:24.642686',
 'sources': {'n_unique_sequences': {'AMPDB': 2689,
   'BIOPEP-UWM': 10,
   'CICERON': 7,
   'DRAMP': 2648,
   'iAMPCN': 22404,
   'MultiTox': 829,
   'Peptipedia2.0': 1470}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence_statistics': {'canonical_filter': {'before': 27136, 'after': 26856},
  'length_filter': {'before': 26856, 'after': 22102},
  'length_distribution': {'min': 5, 'max': 70, 'mean': 23.49, 'median': 20.0}},
 'statistics': {'total_sequences_final': 22102,
  'positive': {'positive_and_unlabel': 574, 'only_positive': 574},
  'negative': {'negative_and_unlabel': 20434, 'only_negative': 20434},
  'only_unlabel': 0,
  'ambiguous': {'n_sequences': 1094,
   'category_pbb': {'description': 'Distribution of ambiguous sequences based on the percentage of positive annotations across sources',
    'categories_definition': 'Bin

- Exporting data

In [39]:
os.makedirs(output_folder, exist_ok=True)

In [40]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [41]:
negative.shape

(20434, 19)

In [42]:
only_negative.shape

(20434, 19)

In [43]:
positive.shape

(574, 19)

In [44]:
only_positive.shape

(574, 19)

In [45]:
only_unlabel.shape

(0, 19)

In [46]:
df_ambiguous.shape

(1094, 20)

In [47]:
negative.to_csv(f"{output_folder}/negative.csv", index=False)

In [48]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)

In [49]:
df_ambiguous.to_csv(f"{output_folder}/ambiguous_data.csv", index=False)